In [6]:
# CIS-117 Lab 3
# CSV file reader module
# Authors: Leslie Fong
# Group 3
# Partner Han Bhone Hset
# Date: 11/08/2026
# We split the read and write portions as read = Leslie, write = Han
# Looking at the openstax chapter section on CSV file, we agreed there
# should be a class objects returned for the data storage hand off exchanged.
#
"""Reading and initial storage and processing of a CSV file.

Open a CSV file for reading the assignment's expected format (has header).
Caller can choose to specify a different file/
We'll also consider to read one web github file when the file path
choices are not present. That only occurs when option skip_url=False.

The expected countries with the assignment's CSV data file only occupy 11 regions,
including one (6th) being unnamed (for Antarctica).
"""

import urllib.request
import csv

# Define a custom row class to store the CSV per row data vs using standard containers directly

class csvRow():
    def __init__(self):
        self.source = ""            # the raw text data defining this instance
        self.row_num = -1       # its original line number in file
        self.header = ""            # reference to the file's header text
        self.vals = []
        self.primary_index = -1     # There is a primary field (ID) for the data, often in position [0]
        self.secondary_index = -1   # The task's major magnet field, but secondary to ID
        self.fields  = {}             # optional dict for accessing fields by key (string)

    # Using a slice defends this code from crashing on accessing an out of range index.
    #print provides only a limited view of the data set, for faster easier development.
    def __repr__(self):
        safe = self.secondary_index
        return f"csvRow: {self.row_num:3d}: {self.vals[safe: safe+1]} => {self.source}\n"
        # return f"csvRow: {self.row_num:3d}: {self.vals[safe: safe + 1]} => {self.source[0:60]}...\n"

class csvDataBase():
    def __init__(self, source = ''):
        self.source = source           # the file (string) which defined the database location
        self.header = None              # reference to the file's header (first row) text fields
        self.raw_header = ""            # raw text of the file's header
        self.primary_index = -1       # There is a primary field (ID) for row data, often in position [0]
        self.secondary_index = -1    # The task's (minor) magnet for reporting processing field.
        self.primary_field = ''           # Lab 3 assignment expects ID as "name"
        self.secondary_field = ''        # Lab 3 assignment task expects grouping by "region"
        self.csv_store  = {}              # list of csvRow for database, in file's order (minus header)
        self.major_fields = {}          # List tree of minor key values which own major keys' values.
        self.minor_dict = {}            # A dict of all rows grouped by the minor key values.
        self.major_dict = {}

    # Generate a dict (to reuse) for searching DB for a field having specific field value,
    # singleton use. Thus any duplicates emit a warning, and only the last one gets saved/
    def getFieldCSVSingle(self, header_value):
        fly = {}
        if header_value in self.header:
            for csv in self.csv_store:
                fval = csv.fields[header_value]
                # print("Processing ", fval)
                if fval in fly:
                    print(f"Warning: duplicate entry found for for field '{header_value}'", csv)
                fly[fval] = csv
        return fly

    # Generate a dict (to reuse) for searching DB for a field having some specific field value
    # Multiple csvRow entries can be stored in the list for a client querying the dict.
    def getFieldCSVMulti(self, header_value):
        fly = {}
        if header_value in self.header:
            for csv in self.csv_store:
                fval = csv.fields[header_value]
                # print("Processing ", fval)
                if fval not in fly:
                    fly[fval] = [csv]
                else:
                    fly[fval].append(csv)
        return fly

    # Uses the getFieldCSVMulti method to store all primaries
    # and returns only one matching row (last one).
    def getMajorCSV(self, value):
        primary = self.primary_field
        if len(self.major_dict) == 0:
            upd = self.getFieldCSVMulti(primary)
            if len(upd) > 0:
                self.major_dict.update(upd)
        safe = self.major_dict.get(value, None)
        if safe == None or len(safe) < 1:
            return csvRow()
        return safe[-1]

    def __repr__(self):
        return f"csvDataBase for {self.source} holding\n{str(self.csv_store)}\n\
" + f"majors: {str(self.major_fields)}\nsecondary view: {str(self.minor_dict)}"


def _cleanQuotes(strq):
    dquote = '"'
    dlen = len(strq)
    if dlen < 2 or strq.find(dquote) < 0:
        return strq
    newstr = ""
    skip = 0
    for n in range(dlen - 1):
        if skip > 0:
            skip = 0
            continue
        char1, char2 = strq[n], strq[n+1]
        if char1 == char2 == dquote:
            newstr += char1
            skip = 1
            continue
        newstr += char1
    # Append the last character if it wasn't skipped
    if skip == 0 and dlen > 0:
        newstr += strq[-1]
    return newstr

# Open the targeted CSV file for reading specific content fields are expected.
# Preserve all information available to facilitate the callers' next writing steps.
# Returns a csvDataBase when reading was successful, otherwise None.

def readCountryCSV(file_path = "", main_field="name", group_field = "region",
                                skip_url=True, verbose=False, verboseErrors = True):
    '''
    Read a predetermined choice of location for a CSV data containing geophysical locations
    and country names.

    Returns a custom class containing the rows of text contained in the file, as well
    as data structures based on their analysis  Analysis is based on 2 of the CSV fields,
    which are listed in the first row "header" - name and region.

    The purpose of the data extracted is intended to write new files of data being
    separated to be recollected by their regions are the same areas..
    '''
    # Here are the file paths we'll try to open after caller's argument choice
    default_paths = ("country_full.csv",
                        "/content/gdrive/MyDrive/Colab Notebooks/country_full.csv")
    # assignments URL at "https://smccd.instructure.com/courses/73989/files/16316539/download?download_frd=1"
    # doesn't provide a publc and universal access to the file content.
    fallback_URL = 'https://raw.githubusercontent.com/LeslieF-smccd/url_data/refs/heads/main/country_full.csv'
    # From the githubs main page at "https://github.com/LeslieF-smccd/url_data/blob/main/country_full.csv"

    key1 = main_field # "name"   # determines the primary_index
    key2 = group_field # "sub-region"  # determines the secondary_index
    paths = list(default_paths)
    datasource = ""
    raw_header = ""
    csv_rows = []

    row_limit = 0   # optional development - limit of number or row lines to process, 0 is unlimited.

    if len(file_path) > 0:
        if file_path not in default_paths or file_path != default_paths[0]:
            if verbose:
                print("Accepting user path", file_path)
            paths.insert(0, file_path)

    # Loop over typical installation's paths for finding the country CSV file
    for times in range(len(paths)):
        path = paths[times]
        try:
            file_obj = None
            file_obj = open(path)
        except FileNotFoundError as e:
            if verbose or verboseErrors:
                print("Open path not found", e)
            file_obj = None
        except PermissionError as e:
            if verbose or verboseErrors:
                print("Open path failed from permissions", e)
            file_obj = None

        if file_obj != None:
            if verbose:
                print("Successfully opened", path)
            datasource = path
            break

    if file_obj != None:
        # Rows separated by newline \n characters can use readlines() to read in
        #  all rows returned as a list of strings
        csv_rows = file_obj.readlines() if file_obj != None else []
        file_obj.close()
        file_obj = "done"  # Prevent reuse as a file, without confusions as None.
    elif not skip_url:
        urls = [fallback_URL]
        if verbose or verboseErrors:
            print("Can't process file open() without access to a file.  Trying a URL backup.")
        for u in urls:
            try:
                byte_text = urllib.request.urlopen(fallback_URL).read()
            except urllib.error.URLError as e:
                if verbose or verboseErrors:
                    print("URL service timeout: ", e, '\n', u, '\n', sep='')
                continue
            except:
                if verbose or verboseErrors:
                    print("URL is not working for:", u)
                continue

            csv_text = byte_text.decode("utf-8")
            csv_rows = csv_text.split('\n') # sensible data has >= 2 line rows
            if len(csv_rows) > 1:
                if verbose:
                    print("URL substitute for read of file worked!\n")
                datasource = u
                break

        if len(datasource) == 0:
            if verbose or verboseErrors:
                print("All predefined data URLs failed.\n")

    else:
        if verbose or verboseErrors:
            print("Can't process file open() without access to a file.")
        return None # csvDataBase()

    collect_rows = []
    header = None
    primary_index, secondary_index = -1, -1

    row_count = -1

    # Process each line-row and split by comma and save data into a 2D structure
    for row in csv_rows:
        # Remove \n character from the data, encountered by file open API
        row = row.strip("\n")
        row_count += 1

        precells = []
        dquote = '"'
        comma = ','
        srow = row
        # Can't split CSV fields by depending only on commas. Invalid assumption for our data's tests.
        while len(srow) > 0:
            if  (next_dq := srow.find(dquote)) < 0: # no quotes left case
                precells.extend(srow.split(comma))
                break

            if next_dq == 0:    # begin quoted str
                srow = srow[1:]
                next_db = srow.find(dquote + comma)
                if next_db >= 0:    # pull out the string minus dq ends
                    precells.append(_cleanQuotes(srow[:next_db]))
                    # print(f"Sliced out precells[-1]:  >{precells}<  len={len(precells)}")
                    srow = srow[next_db+2:]
                    continue
                next_dq = srow.find(dquote)
                if next_dq >=  0:     # Else malformed quotes in input!
                    srow = srow[:-1]
                precells.append(_cleanQuotes(srow))
                break;

            # next_dq > 0
            precells.extend(srow[:next_dq-1].split(comma))
            srow = srow[next_dq:]

        cells = precells

        # print(f"\n split {len(cells)} cells with dquote =", cells)

        # The supplied data format provides first row as a header for describing the other rows' values.
        # Confirm the data as having the expected CSV pair of fields in order to proceed.
        if row_count == 0:
            header = cells
            if {key1, key2} <= set(header) :
                primary_index = header.index(key1)
                secondary_index = header.index(key2)
                raw_header = row
                continue

            else:
                print("Failed to find expected fields in data's CSV header!")
                return None # csvDataBase()

        csvr = csvRow() # custom class for saving key data of each data row, excluding header
        csvr.row_num = row_count
        csvr.primary_index = primary_index
        csvr.secondary_index = secondary_index
        csvr.main_field = key1
        csvr.group_field = key2
        csvr.source = row
        csvr.vals = cells.copy()
        csvr.header = header
        if type(csvr.header) != str:
            csvr.header = ""
        if len(header) >= 2:
            csvr.fields = dict( zip(header, cells) )
        collect_rows.append(csvr)

        if row_limit and lines_processed >= row_limit: # optional to test with smaller subset of data
            break

    return_db = csvDataBase(datasource)

    # Build a secondary/minor value based dictionary of the rows' primary values only
    minor_lookup = {}
    for n in range(len(collect_rows)):
        kv1 = collect_rows[n].fields[key1]
        kv2 = collect_rows[n].fields[key2]
        if kv2 not in minor_lookup:
            minor_lookup[kv2] = []
        if kv1 not in minor_lookup[kv2]:
            minor_lookup[kv2].append(kv1)

    # Build a secondary/minor key based dict of each entire row having the minor key's value.
    secondary_regrouped = {}
    num_rows = len(collect_rows)
    for keys2 in  minor_lookup.keys():
        secondary_regrouped[keys2] = [collect_rows[m] for m in range(num_rows
                                        ) if collect_rows[m].vals[secondary_index] == keys2]

    # Assemble the data collected in a returned database object.
    return_db.header = header                # shared reference to the file's header (first row) text fields
    return_db.raw_header = raw_header  # text string of the header. without its newline
    return_db.primary_index = primary_index
    return_db.secondary_index = secondary_index
    return_db.primary_field = key1
    return_db.secondary_field = key2
    return_db.csv_store  = collect_rows
    return_db.major_fields = minor_lookup
    return_db.minor_dict = secondary_regrouped
    return return_db


In [7]:
# testing of functionality as a standalone file's program
# The print functionality by __repr__ is not intended for  non-dev end users.

import csv

if __name__ == "__main__":
    # quick and dirty write validater
    def writerman(file_base, data):
        with open('test_' + str(file_base) + '.csv', 'w', newline='') as csvfile:
            fieldnames = ['name', 'region']
            writer = csv.DictWriter(csvfile, fieldnames=fieldnames, quoting=csv.QUOTE_MINIMAL)
            writer.writeheader()
            writer.writerows(data)
    True

    verbose = False
    csvdb = readCountryCSV("country_full.csv", skip_url = False, verboseErrors = True)
    # other options to consider:  main_field = "name", group_field = "region", verbose=verbose

    if verbose:
        print(csvdb)
    if csvdb != None:
        minors = list(csvdb.major_fields.keys())
        if len(minors) > 0:
            print(f"Read OK")
            print(f"Data file comes from", csvdb.source)
            print(f"\nReceived data of {len(minors)} {csvdb.secondary_field}s:", *list(minors))
            print("CSV DB Fields:", *csvdb.header)
            print(f"The DB common header text is:", [csvdb.raw_header])
            print("Example row:",  csvdb.csv_store[0])
            for n in ['United States of America',  'American Samoa','China', 'France']:
                print( f"Queried getMajorCSV('{n}') gets: ", csvdb.getMajorCSV(n), end='')

            # Example of data built for CSV DictWriter functions: list of dicts
            key1 = 'name'
            key2 = 'region'
            for regionName in csvdb.minor_dict:
                print("\nWrite a CSV file for ", regionName)
                data = [ {key1: row.fields[key1], key2: row.fields[key2]} for
                                row in csvdb.minor_dict[regionName] ]
                print("Call writer on", data, f"\nwith fieldnames as {[key1, key2]}'")
                fname = regionName if len(regionName) > 0 else "undefined_ant"
                writerman(fname, data)
None

Open path not found [Errno 2] No such file or directory: 'country_full.csv'
Open path not found [Errno 2] No such file or directory: '/content/gdrive/MyDrive/Colab Notebooks/country_full.csv'
Can't process file open() without access to a file.  Trying a URL backup.
Read OK
Data file comes from https://raw.githubusercontent.com/LeslieF-smccd/url_data/refs/heads/main/country_full.csv

Received data of 6 regions: Asia Europe Africa Oceania Americas 
CSV DB Fields: name alpha-2 alpha-3 country-code iso_3166-2 region sub-region intermediate-region region-code sub-region-code intermediate-region-code
The DB common header text is: ['name,alpha-2,alpha-3,country-code,iso_3166-2,region,sub-region,intermediate-region,region-code,sub-region-code,intermediate-region-code']
Example row: csvRow:   1: ['Asia'] => Afghanistan,AF,AFG,004,ISO 3166-2:AF,Asia,Southern Asia,"",142,034,""

Queried getMajorCSV('United States of America') gets:  csvRow: 236: ['Americas'] => United States of America,US,USA,840